# SVM vs QKSVM — Comparison

This notebook loads pre-computed results from:
- `../classical/svm_results.json`
- `../quantum/qksvm_results.json`

and generates all comparison plots and the summary table.

**No model is trained here.** Re-run freely without touching the training notebooks.

Set `COMPARE_4F = True` once the 4-feature result files are also available.

## Imports

In [4]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## Configuration

In [ ]:
COMPARE_4F = False

SVM_FILE     = "../classical/svm_results.json"
QKSVM_FILE   = "../quantum/qksvm_results.json"
SVM_4F_FILE  = "../classical/svm_results_4f.json"
QKSVM_4F_FILE= "../quantum/qksvm_results_4f.json"

for f in [SVM_FILE, QKSVM_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(
            f"Missing results file: {f}\n"
            "Run svm.ipynb and qksvm.ipynb first."
        )

if COMPARE_4F:
    for f in [SVM_4F_FILE, QKSVM_4F_FILE]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Missing 4-feature results file: {f}")

print("Results files found — ready to compare.")

FileNotFoundError: Missing results file: ../classification/quantum/qksvm_results.json
Run svm.ipynb and qksvm.ipynb first.

## Load Results

In [ ]:
with open(SVM_FILE) as f:
    svm = json.load(f)

with open(QKSVM_FILE) as f:
    qksvm = json.load(f)

if COMPARE_4F:
    with open(SVM_4F_FILE) as f:
        svm4 = json.load(f)
    with open(QKSVM_4F_FILE) as f:
        qksvm4 = json.load(f)

print("SVM results:")
print(f"  Test accuracy : {svm['accuracy_test']:.4f}")
print(f"  Train time    : {svm['train_time_s']:.4f} s")
print()
print("QKSVM results:")
print(f"  Test accuracy (sim)   : {qksvm['accuracy_test_sim']:.4f}")
if qksvm.get('real_hw_run'):
    print(f"  Test accuracy (real)  : {qksvm['accuracy_test_real']:.4f}")
print(f"  Kernel compute time   : {qksvm['total_kernel_time_s']:.2f} s")
print(f"  SVM train time        : {qksvm['svm_train_time_s']:.4f} s")
print(f"  Total time            : {qksvm['total_time_s']:.2f} s")

## Plot 1 — Accuracy Comparison

In [ ]:
labels = ["SVM\n(RBF kernel)", "QKSVM\n(simulator)"]
values = [svm["accuracy_test"], qksvm["accuracy_test_sim"]]
colors = ["tab:blue", "tab:orange"]
yerr   = [svm["cv_std"], 0]

if qksvm.get("real_hw_run") and qksvm.get("accuracy_test_real") is not None:
    labels.append("QKSVM\n(real HW)")
    values.append(qksvm["accuracy_test_real"])
    colors.append("tab:red")
    yerr.append(0)

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(x, values, color=colors, width=0.5,
              yerr=yerr, capsize=5, error_kw={"elinewidth": 1.5})

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Test Accuracy")
ax.set_title(f"SVM vs QKSVM — Test Accuracy ({svm['n_features']} features)")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="Perfect accuracy")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig("plot1_accuracy.png", dpi=150)
plt.show()

## Plot 2 — Time Breakdown

For the QKSVM, total time splits into two components: kernel computation (quantum)
and SVM training (classical). This breakdown shows where the cost actually lies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: total training time comparison
t_labels = ["SVM\n(RBF kernel)", "QKSVM\n(total)"]
t_values = [svm["train_time_s"], qksvm["total_time_s"]]
t_colors = ["tab:blue", "tab:orange"]

bars = axes[0].bar(t_labels, t_values, color=t_colors, width=0.4)
for bar, val in zip(bars, t_values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val * 1.1,
                 f"{val:.3f} s", ha="center", va="bottom", fontsize=10)
axes[0].set_yscale("log")
axes[0].set_ylabel("Time (seconds, log scale)")
axes[0].set_title("Total time comparison")
axes[0].grid(axis="y", alpha=0.4)

# Right: QKSVM time breakdown (kernel vs SVM)
breakdown_labels = ["Kernel\ncomputation", "SVM training\n(classical)"]
breakdown_values = [qksvm["total_kernel_time_s"], qksvm["svm_train_time_s"]]
breakdown_colors = ["tab:orange", "tab:green"]

bars2 = axes[1].bar(breakdown_labels, breakdown_values, color=breakdown_colors, width=0.4)
for bar, val in zip(bars2, breakdown_values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val * 1.1,
                 f"{val:.3f} s", ha="center", va="bottom", fontsize=10)
axes[1].set_yscale("log")
axes[1].set_ylabel("Time (seconds, log scale)")
axes[1].set_title("QKSVM time breakdown")
axes[1].grid(axis="y", alpha=0.4)

plt.suptitle(f"Training Time — {svm['n_features']} features")
plt.tight_layout()
plt.savefig("plot2_time_breakdown.png", dpi=150)
plt.show()

## Plot 3 — Confusion Matrices Side by Side

In [ ]:
cm_svm   = confusion_matrix(svm["y_test"],   svm["y_pred_test"])
cm_qksvm = confusion_matrix(qksvm["y_test"], qksvm["y_pred_test_sim"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in zip(axes,
                          [cm_svm, cm_qksvm],
                          ["SVM (RBF kernel)", "QKSVM (simulator)"]):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=["Versicolor", "Virginica"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(title)

plt.suptitle(f"Confusion Matrices — {svm['n_features']} features", y=1.02)
plt.tight_layout()
plt.savefig("plot3_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot 4 — 2-feature vs 4-feature Comparison (optional)

In [ ]:
if COMPARE_4F:
    experiments = [
        ("SVM 2f",   svm["accuracy_test"],      svm["train_time_s"],      "tab:blue"),
        ("SVM 4f",   svm4["accuracy_test"],     svm4["train_time_s"],     "tab:cyan"),
        ("QKSVM 2f", qksvm["accuracy_test_sim"], qksvm["total_time_s"],   "tab:orange"),
        ("QKSVM 4f", qksvm4["accuracy_test_sim"],qksvm4["total_time_s"],  "tab:red"),
    ]

    names  = [e[0] for e in experiments]
    accs   = [e[1] for e in experiments]
    times  = [e[2] for e in experiments]
    colors = [e[3] for e in experiments]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].bar(names, accs, color=colors)
    for i, val in enumerate(accs):
        axes[0].text(i, val + 0.01, f"{val:.3f}", ha="center", fontsize=9)
    axes[0].set_ylim(0, 1.15)
    axes[0].set_ylabel("Test Accuracy")
    axes[0].set_title("Test Accuracy: 2 features vs 4 features")
    axes[0].grid(axis="y", alpha=0.4)

    axes[1].bar(names, times, color=colors)
    axes[1].set_yscale("log")
    axes[1].set_ylabel("Total time (s, log scale)")
    axes[1].set_title("Total Time: 2 features vs 4 features")
    axes[1].grid(axis="y", alpha=0.4)

    plt.tight_layout()
    plt.savefig("plot4_2f_vs_4f.png", dpi=150)
    plt.show()
else:
    print("COMPARE_4F is False — skipping 2f vs 4f comparison.")
    print("Set COMPARE_4F = True once both 4-feature result files are available.")

## Summary Table

In [ ]:
summary = {
    "n_features"                  : svm["n_features"],
    "n_train"                     : svm["n_train"],
    "n_test"                      : svm["n_test"],
    # SVM
    "svm_accuracy_test"           : svm["accuracy_test"],
    "svm_cv_mean"                 : svm["cv_mean"],
    "svm_cv_std"                  : svm["cv_std"],
    "svm_train_time_s"            : svm["train_time_s"],
    "svm_n_support_vectors"       : svm["n_support_vectors"],
    # QKSVM
    "qksvm_accuracy_test_sim"     : qksvm["accuracy_test_sim"],
    "qksvm_accuracy_test_real"    : qksvm.get("accuracy_test_real"),
    "qksvm_kernel_train_time_s"   : qksvm["kernel_train_time_s"],
    "qksvm_kernel_test_time_s"    : qksvm["kernel_test_time_s"],
    "qksvm_svm_train_time_s"      : qksvm["svm_train_time_s"],
    "qksvm_total_time_s"          : qksvm["total_time_s"],
    "qksvm_n_qubits"              : qksvm["n_qubits"],
    "qksvm_n_trainable_params"    : qksvm["n_trainable_params"],
    "qksvm_n_support_vectors"     : qksvm["n_support_vectors"],
    "qksvm_real_hw_run"           : qksvm.get("real_hw_run", False),
    "qksvm_backend"               : qksvm.get("backend_name"),
}

if COMPARE_4F:
    summary["svm_4f_accuracy_test"]      = svm4["accuracy_test"]
    summary["svm_4f_train_time_s"]       = svm4["train_time_s"]
    summary["qksvm_4f_accuracy_test_sim"]= qksvm4["accuracy_test_sim"]
    summary["qksvm_4f_total_time_s"]     = qksvm4["total_time_s"]

with open("comparison_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Print as readable table
col_w = 32
print(f"{'Metric':<{col_w}} {'SVM':>14} {'QKSVM (sim)':>14} {'QKSVM (real)':>14}")
print("-" * (col_w + 46))

rows = [
    ("Test accuracy",
     f"{svm['accuracy_test']:.4f}",
     f"{qksvm['accuracy_test_sim']:.4f}",
     f"{qksvm['accuracy_test_real']:.4f}" if qksvm.get('accuracy_test_real') else "N/A"),
    ("Train accuracy",
     f"{svm['accuracy_train']:.4f}",
     f"{qksvm['accuracy_train_sim']:.4f}",
     "N/A"),
    ("CV mean ± std",
     f"{svm['cv_mean']:.4f} ± {svm['cv_std']:.4f}",
     "N/A",
     "N/A"),
    ("Kernel compute time (s)",
     "—",
     f"{qksvm['total_kernel_time_s']:.2f}",
     f"{qksvm['kernel_real_time_s']:.2f}" if qksvm.get('kernel_real_time_s') else "N/A"),
    ("SVM train time (s)",
     f"{svm['train_time_s']:.4f}",
     f"{qksvm['svm_train_time_s']:.4f}",
     "—"),
    ("Total time (s)",
     f"{svm['train_time_s']:.4f}",
     f"{qksvm['total_time_s']:.2f}",
     "N/A"),
    ("Support vectors",
     str(svm['n_support_vectors']),
     str(qksvm['n_support_vectors']),
     "—"),
    ("Trainable parameters",
     "—",
     str(qksvm['n_trainable_params']),
     "—"),
]

for label, sv, qk, qr in rows:
    print(f"{label:<{col_w}} {sv:>14} {qk:>14} {qr:>14}")

print()
print("Summary saved to comparison_summary.json")